# RQ2, emotion specialist: Emotion-LLaMA on CH-SIMS v2 (Kaggle, 2× T4)

Label scoring with Emotion-LLaMA (Cheng et al., NeurIPS 2024; Llama-2-7B-chat + LoRA, EVA-CLIP image
encoder, HuBERT audio features), the one non-Qwen model trained for multimodal emotion recognition.
All 4,403 clips × 4 conditions, written to `/kaggle/working/emollama_runs/scores.jsonl` in the format of the other runs.

**Environment.** The repository needs Python 3.9 (torch 2.0, transformers 4.30, peft 0.2.0); Kaggle runs 3.13.
This notebook builds a separate Python 3.9 environment with `uv`, installs the repository's pins, and runs
`score_emotion_llama.py` in it. The model is built by the authors' own code (demo config), unchanged.

**What the model sees** (details in the script's docstring):
- the clip's first extracted frame (the demo uses the first frame), HuBERT features of the audio (as the demo),
  zeros for the FaceMAE and VideoMAE features (as the demo: the authors publish no extraction for raw clips);
- the evaluation instruction of the authors' dataset code, with the Chinese transcript;
- the checkpoint's six labels in the order of its training instruction: happy, sad, neutral, angry, worried, surprise.
  "worried" is recorded as Fear (the authors' zero-shot alias). **There is no Disgust**: `logit_Disgust` is empty,
  and the comparison with humans uses the six labels.
- T, A and V alone were never seen in training (T: no media; A: only the three feature tokens; V: audio feature zero).

**Weights** (pinned revisions): the demo checkpoint `Emoation_LLaMA.pth` from the authors' Hugging Face Space,
Llama-2-7b-chat from the mirror the Space itself uses, `TencentGameMate/chinese-hubert-large`, and EVA-CLIP-g,
which the repository's code downloads. Licences: the repository is CC BY-NC 4.0 plus the Llama 2 licence; research use.

**Precision and memory.** As in the authors' evaluation config: Llama in fp16 (`low_resource: False`), split over
the two GPUs; EVA in fp16; HuBERT in fp32. The script checks that every tensor of the checkpoint was loaded
(the authors' loader uses `strict=False`).

**Running it**
1. *Accelerator*: **GPU T4 x2**; *Internet*: **on**; *Add input*: the preprocessed `chsims-mm` dataset.
2. First set `LIMIT = 10` and run all interactively (about 25 GB of downloads): check the checkpoint line,
   the label tokens, the prompts, the `label mass` and the timing lines, and send them over before the full run.
3. Full run: `LIMIT = None`, **Save Version → Save & Run All (Commit)**. It stops after `TIME_BUDGET_H`;
   to continue, add this notebook's output as an input and commit again.

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

TIME_BUDGET_H = 11.0         # stop starting new items after this many hours (counted from the start of the notebook)
LIMIT = None                 # e.g. 10 for a quick check: 10 test clips + 10 other clips
SPLIT_LAYER = 14             # Llama layers 0..13 on GPU 0 (with EVA), the rest on GPU 1 (with HuBERT)
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/emollama_runs"))
WORK = Path(os.environ.get("EMOLLAMA_WORK", "/tmp/emollama"))  # environment, code, weights: not part of the output
ENV_PY = Path(os.environ.get("EMOLLAMA_PY", WORK / "env" / "bin" / "python"))
REPO = Path(os.environ.get("EMOLLAMA_REPO", WORK / "Emotion-LLaMA"))
REPO_COMMIT = "6f8d05087b2a91e445ad986545977fda7eb1b6a4"           # main on 2026-10-18
LLAMA_REPO, LLAMA_REV = "ZebangCheng/Emotion-LLaMA", "2ba523d3543da46b9f8b6dab308f2a56298c9e37"
SPACE_REPO, SPACE_REV = "ZebangCheng/Emotion-LLaMA", "9cc18338a896d6aa6a3090bbf37055bf93909b73"
HUBERT_REPO, HUBERT_REV = "TencentGameMate/chinese-hubert-large", "90cb660492214f687e60f5ca509b20edae6e75bd"
MODEL_NAME = "ZebangCheng/Emotion-LLaMA (Emoation_LLaMA.pth)"
T_START = time.time()

DATA_DIR = os.environ.get("CHSIMS_MM_DIR")
if DATA_DIR is None:
    hits = [h for k in range(6) for h in sorted(glob.glob("/kaggle/input/" + "*/" * k + "manifest.csv"))]  # no ** : media folders make it slow
    assert hits, "No manifest.csv under /kaggle/input: attach the preprocessed dataset."
    DATA_DIR = os.path.dirname(hits[0])
DATA_DIR = Path(DATA_DIR)
DRY_RUN = False
WORK.mkdir(parents=True, exist_ok=True)


def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("data:", DATA_DIR, "| out:", OUT_DIR, "| dry run:", DRY_RUN)
print("free disk on / (GB):", round(shutil.disk_usage("/").free / 1e9, 1))
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i}", torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1), "GiB")

## Python 3.9 environment and the repository

In [ ]:
PINS = ['huggingface-hub==0.18.0', 'pyyaml==6.0', 'regex==2022.10.31', 'tokenizers==0.13.2', 'tqdm==4.64.1', 'transformers==4.30.0', 'timm==0.6.13', 'webdataset==0.2.48', 'omegaconf==2.3.0', 'opencv-python==4.7.0.72', 'decord==0.6.0', 'peft==0.2.0', 'accelerate==0.20.3', 'iopath==0.1.10', 'matplotlib==3.7.0', 'psutil==5.9.4', 'wandb==0.16.0', 'moviepy==1.0.3', 'soundfile==0.12.1', 'sentencepiece==0.2.2', 'numpy==1.26.4', 'pandas==2.3.3', 'setuptools==69.5.1']
if not ENV_PY.exists():
    sh(f"{sys.executable} -m pip install -q uv")
    sh(f"{sys.executable} -m uv venv -q --python 3.9 {ENV_PY.parent.parent}")  # uv fetches a Python 3.9 build
    sh(f"{sys.executable} -m uv pip install -q --python {ENV_PY} torch==2.0.0 torchvision==0.15.1 torchaudio==2.0.1 "
       "--index-url https://download.pytorch.org/whl/cu118")
    sh(f"{sys.executable} -m uv pip install -q --python {ENV_PY} " + " ".join(f'"{p}"' for p in PINS))
sh(f"{ENV_PY} -c 'import sys, torch, transformers, peft; print(sys.version.split()[0], torch.__version__, "
   "transformers.__version__, peft.__version__, \"cuda:\", torch.cuda.is_available())'")
if not REPO.exists():
    sh(f"git clone -q https://github.com/ZebangCheng/Emotion-LLaMA.git {REPO}")
    sh(f"git -C {REPO} checkout -q {REPO_COMMIT}")
sh(f"git -C {REPO} log -1 --format='%H %cd'")

## Weights

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

if os.environ.get("EMOLLAMA_LOCAL"):  # local test with small stand-ins
    LLAMA, CKPT, HUBERT = os.environ["EMOLLAMA_LLAMA"], os.environ["EMOLLAMA_CKPT"], os.environ["EMOLLAMA_HUBERT"]
else:
    t0 = time.time()
    LLAMA = snapshot_download(LLAMA_REPO, revision=LLAMA_REV, local_dir=WORK / "llama", allow_patterns=[
        "config.json", "generation_config.json", "model.safetensors.index.json", "model-*.safetensors",
        "tokenizer.model", "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json"])
    CKPT = hf_hub_download(SPACE_REPO, "Emoation_LLaMA.pth", repo_type="space", revision=SPACE_REV, local_dir=WORK / "ckpt")
    HUBERT = snapshot_download(HUBERT_REPO, revision=HUBERT_REV, local_dir=WORK / "hubert",
                               allow_patterns=["config.json", "preprocessor_config.json", "pytorch_model.bin"])
    print(f"downloaded in {time.time() - t0:.0f} s | free disk on / (GB): {shutil.disk_usage('/').free / 1e9:.1f}")
print(LLAMA, CKPT, HUBERT, sep="\n")

## Scoring script

Runs in the Python 3.9 environment. Kept in this repository as `kaggle/score_emotion_llama.py`; the cell
below writes the same file.

In [ ]:
%%writefile score_emotion_llama.py
"""Emotion-LLaMA label scoring on CH-SIMS v2, inside the repository's own Python 3.9 environment.

Run by kaggle/run_emotion_llama.ipynb (which also builds the environment and downloads the weights).
The model is built with the authors' code (`minigpt4`, demo config) and scored like the other RQ2 runs:
one forward pass, logits of the first token of each label, written as JSON lines.

How the inputs follow the authors' code:
- Image: the clip's first extracted frame through the model's eval image processor (448 × 448), as in the
  demo, which takes the first frame of the video.
- Audio: HuBERT-large (Chinese) last hidden layer, averaged over time, as in the demo.
- FaceMAE and VideoMAE features: zeros, as in the demo (the authors publish no extraction for raw clips).
- Prompt: the evaluation instruction of the training dataset code,
  `<video><VideoHere></video> <feature><FeatureHere></feature> The person in video says: {text}.  [emotion] Please
  determine which emotion label in the video represents: {labels}. ` in the Llama-2 chat wrapper
  `<s>[INST] ... [/INST]`, embedded with the model's `get_context_emb`.
- Labels: the checkpoint's six, in the order of its training instruction (happy, sad, neutral, angry, worried,
  surprise); "worried" is read as Fear, the alias the authors use for zero-shot evaluation. There is no Disgust.

Conditions the model was not trained on (A, T, V) are built as close to the trained input as possible:
- V: image, audio feature set to zeros, no transcript line.
- A: no image tokens (the 256 patch tokens and the class token are left out), only the three feature tokens.
- T: no media placeholders, only the transcript line and the question.
"""

import argparse
import glob
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from PIL import Image

LABELS = ["Anger", "Disgust", "Fear", "Joy", "Neutral", "Sadness", "Surprise"]  # record format of the other runs
MODEL_LABELS = ["happy", "sad", "neutral", "angry", "worried", "surprise"]  # training instruction order
AS_LABEL = {"happy": "Joy", "sad": "Sadness", "neutral": "Neutral", "angry": "Anger", "worried": "Fear",
            "surprise": "Surprise"}
CONDITIONS = ["TAV", "T", "A", "V"]
INSTRUCTION = "Please determine which emotion label in the video represents: {}.".format(", ".join(MODEL_LABELS))
SR = 16000


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--repo", required=True, help="Emotion-LLaMA repository checkout")
    p.add_argument("--data-dir", required=True, help="folder with manifest.csv, frames/ and audio/")
    p.add_argument("--out-dir", required=True)
    p.add_argument("--llama", required=True, help="Llama-2-7b-chat-hf folder")
    p.add_argument("--ckpt", required=True, help="Emotion-LLaMA checkpoint (.pth)")
    p.add_argument("--hubert", required=True, help="chinese-hubert-large folder")
    p.add_argument("--model-name", default="ZebangCheng/Emotion-LLaMA (Emoation_LLaMA.pth)")
    p.add_argument("--resume-glob", default="", help="earlier scores.jsonl files to copy in, e.g. /kaggle/input/**/emollama_runs/scores.jsonl")
    p.add_argument("--split-layer", type=int, default=14, help="Llama layers below this on cuda:0, the rest on cuda:1")
    p.add_argument("--limit", type=int, default=0, help="e.g. 10: 10 test clips + 10 other clips")
    p.add_argument("--time-budget-h", type=float, default=11.0)
    p.add_argument("--cpu-test", action="store_true", help="local check on CPU: fp32 everywhere")
    return p.parse_args()


args = parse_args()
T_START = time.time()
sys.path.insert(0, args.repo)
os.chdir(args.repo)

from minigpt4.common.config import Config  # noqa: E402
from minigpt4.common.registry import registry  # noqa: E402
from minigpt4.datasets.builders import *  # noqa: E402,F401,F403  (registers the builders, as app.py does)
from minigpt4.models import *  # noqa: E402,F401,F403
from minigpt4.processors import *  # noqa: E402,F401,F403
from minigpt4.tasks import *  # noqa: E402,F401,F403
from transformers import HubertModel, Wav2Vec2FeatureExtractor  # noqa: E402

# ---- model ------------------------------------------------------------------------------

options = [f"model.ckpt={args.ckpt}", f"model.llama_model={args.llama}", "model.low_resource=False"]
if args.cpu_test:
    options.append("model.vit_precision=fp32")  # fp16 convolutions do not run on the CPU
cfg = Config(argparse.Namespace(cfg_path="eval_configs/demo.yaml", options=options))
model_cfg = cfg.model_cfg
t0 = time.time()
model = registry.get_model_class(model_cfg.arch).from_config(model_cfg).eval()
vis_cfg = cfg.datasets_cfg.feature_face_caption.vis_processor.train
vis_processor = registry.get_processor_class(vis_cfg.name).from_config(vis_cfg)

# from_config loads the checkpoint with strict=False, so check that every tensor of it really arrived
ckpt = torch.load(args.ckpt, map_location="cpu")["model"]
own = model.state_dict()
n_layers = model.llama_model.base_model.model.config.num_hidden_layers
expected = [k for k in ckpt if ".layers." not in k or int(k.split(".layers.")[1].split(".")[0]) < n_layers]
missing = [k for k in expected if k not in own]
differ = [k for k in expected if k in own and not torch.equal(own[k].float(), ckpt[k].float())]
print(f"checkpoint: {len(expected)} tensors expected, {len(missing)} missing, {len(differ)} different", flush=True)
assert not missing and not differ, (missing[:5], differ[:5])
del ckpt, own

if args.cpu_test:
    model.llama_model.float()  # fp16 matmuls do not run on the CPU in torch 2.0
    inner = model.llama_model.base_model.model
    inner.lm_head = inner.lm_head[0]  # peft's output wrapper casts its input back to fp16
    DEVICE = AUDIO_DEVICE = torch.device("cpu")
else:
    from accelerate import dispatch_model

    DEVICE, AUDIO_DEVICE = torch.device("cuda:0"), torch.device("cuda:1")
    llama = model.llama_model.base_model.model  # LlamaForCausalLM inside the LoRA wrapper
    device_map = {"model.embed_tokens": 0, "model.norm": 1, "lm_head": 1}
    device_map.update({f"model.layers.{i}": 0 if i < args.split_layer else 1 for i in range(n_layers)})
    dispatch_model(llama, device_map=device_map)
    for name in ("visual_encoder", "ln_vision", "llama_proj", "feats_llama_proj1", "feats_llama_proj2",
                 "feats_llama_proj3", "cls_tk_llama_proj", "attention"):
        getattr(model, name).to(DEVICE)
assert model.device == DEVICE, model.device  # maybe_autocast and get_context_emb follow this device

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(args.hubert)
hubert = HubertModel.from_pretrained(args.hubert).eval().to(AUDIO_DEVICE)  # fp32, as in the demo
tokenizer = model.llama_tokenizer
label_first_ids = [tokenizer(w, add_special_tokens=False).input_ids[0] for w in MODEL_LABELS]
assert len(set(label_first_ids)) == len(MODEL_LABELS), label_first_ids
print(f"loaded in {time.time() - t0:.0f} s | label first tokens:",
      {w: tokenizer.convert_ids_to_tokens(i) for w, i in zip(MODEL_LABELS, label_first_ids)}, flush=True)
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} allocated after load: {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB", flush=True)


# ---- inputs -----------------------------------------------------------------------------

@torch.no_grad()
def audio_feature(row):
    samples, sr = sf.read(str(Path(args.data_dir) / row["audio_path"]), dtype="float32")
    assert sr == SR and samples.ndim == 1, (sr, samples.shape)
    input_values = feature_extractor(samples, sampling_rate=sr, return_tensors="pt").input_values.to(AUDIO_DEVICE)
    hidden = hubert(input_values, output_hidden_states=True).hidden_states
    feat = torch.stack(hidden)[[-1]].sum(dim=0)  # last layer, (1, T, 1024): the demo's own lines
    return feat.mean(dim=1, keepdim=True).cpu()  # (1, 1, 1024)


def first_frame(row):
    frames = sorted(glob.glob(str(Path(args.data_dir) / row["frames_dir"] / "*.jpg")))
    return vis_processor(Image.open(frames[0]).convert("RGB")).unsqueeze(0).to(DEVICE)


def feature_tokens(video_features):
    # the three feature tokens exactly as in MiniGPTv2.encode_img, without the image tokens
    with model.maybe_autocast():
        f = video_features.to(DEVICE)
        split = torch.split(f, 1, dim=1)
        return torch.stack([model.feats_llama_proj1(split[0].squeeze(1)), model.feats_llama_proj2(split[1].squeeze(1)),
                            model.feats_llama_proj3(split[2].squeeze(1))], dim=1)


def instruction(row, condition):
    line = f"The person in video says: {row['text']}. " if "T" in condition else ""
    media = "<video><VideoHere></video> <feature><FeatureHere></feature> " if condition != "T" else ""
    return "<s>[INST] " + media + "{} [{}] {} ".format(line, "emotion", INSTRUCTION) + " [/INST]"


@torch.no_grad()
def context_embedding(row, condition):
    text = instruction(row, condition)
    if condition == "T":  # no placeholder: the same tokenization as get_context_emb's first segment
        ids = tokenizer(text, return_tensors="pt", add_special_tokens=True).input_ids.to(DEVICE)
        return model.embed_tokens(ids), text
    zeros = torch.zeros([1, 2, 1024])  # FaceMAE and VideoMAE, zero as in the demo
    audio = audio_feature(row) if "A" in condition else torch.zeros([1, 1, 1024])
    video_features = torch.cat((zeros, audio), dim=1)
    if "V" in condition:
        media, _ = model.encode_img(first_frame(row), video_features)
    else:
        media = feature_tokens(video_features)
    return model.get_context_emb(text, [media]), text


@torch.no_grad()
def label_logits(embs):
    with model.maybe_autocast():
        out = model.llama_model(inputs_embeds=embs, use_cache=False)
    return out.logits[0, -1].float()


# ---- run --------------------------------------------------------------------------------

out_dir = Path(args.out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
SCORES = out_dir / "scores.jsonl"
manifest = pd.read_csv(Path(args.data_dir) / "manifest.csv", dtype={"video_id": str, "clip_id": str})
clips = manifest[manifest["status"] == "ok"].reset_index(drop=True)
if args.limit:
    clips = pd.concat([clips[clips["mode"] == "test"].head(args.limit), clips[clips["mode"] != "test"].head(args.limit)])
print(len(clips), "clips to score", flush=True)

row0 = clips.iloc[0]
print("prompt (T+A+V):", repr(instruction(row0, "TAV")), flush=True)
print("prompt (T):", repr(instruction(row0, "T")), flush=True)

# the T+A+V and V prompts must equal what the authors' evaluation builds (dataset format + conversation template)
from minigpt4.conversation.conversation import CONV_VISION_minigptv2  # noqa: E402
from minigpt4.evaluation.prompting import prepare_conversation_texts  # noqa: E402

conv = CONV_VISION_minigptv2.copy()
conv.system = ""
for condition, line in (("TAV", f"The person in video says: {row0['text']}. "), ("V", "")):
    dataset_text = "<video><VideoHere></video> <feature><FeatureHere></feature> {} [{}] {} ".format(line, "emotion", INSTRUCTION)
    assert prepare_conversation_texts([dataset_text], conv)[0] == instruction(row0, condition), condition

# the label read from the logits must match the first word of the authors' own greedy answer (T+A+V)
for _, row in clips.head(3).iterrows():
    embs, _ = context_embedding(row, "TAV")
    logits = label_logits(embs)
    video_features = torch.cat((torch.zeros([1, 2, 1024]), audio_feature(row)), dim=1)
    answer = model.generate(first_frame(row), video_features, [instruction(row, "TAV")], max_new_tokens=8)[0]
    print(f"{row['uid']}: authors' generate -> {answer!r} | top token {tokenizer.convert_ids_to_tokens(int(logits.argmax()))!r} | "
          f"top of the six: {MODEL_LABELS[int(logits[label_first_ids].argmax())]}", flush=True)


def key(rec):
    return (rec["model"], rec["uid"], rec["condition"], rec["task"])


sources = sorted(set(glob.glob(args.resume_glob, recursive=True) if args.resume_glob else []) | ({str(SCORES)} if SCORES.exists() else set()))
records = {}
for src in sources:
    for line in open(src, encoding="utf-8"):
        if line.strip() and "error" not in (rec := json.loads(line)):
            records[key(rec)] = rec
with open(SCORES, "w", encoding="utf-8") as f:
    for rec in records.values():
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
done = set(records)
print(f"{len(records)} finished records from {len(sources)} file(s)", flush=True)

jobs = [(row, c) for c in CONDITIONS for _, row in clips.iterrows()]
todo = [j for j in jobs if (args.model_name, j[0]["uid"], j[1], "score") not in done]
print(f"{len(jobs)} items, {len(jobs) - len(todo)} already done, {len(todo)} to run", flush=True)

n_done, stopped_early, t_run = 0, False, time.time()
for row, condition in todo:
    if (time.time() - T_START) / 3600 > args.time_budget_h:
        stopped_early = True
        break
    rec = {"model": args.model_name, "uid": row["uid"], "condition": condition, "task": "score"}
    try:
        t0 = time.perf_counter()
        embs, _ = context_embedding(row, condition)
        t1 = time.perf_counter()
        logits = label_logits(embs)
        rec.update({"n_tokens": int(embs.shape[1]), "prep_s": round(t1 - t0, 3), "gpu_s": round(time.perf_counter() - t1, 3),
                    "finite": bool(torch.isfinite(logits).all()), "lse": float(torch.logsumexp(logits, dim=-1))})
        rec.update({f"logit_{l}": None for l in LABELS})
        rec.update({f"logit_{AS_LABEL[w]}": float(logits[i]) for w, i in zip(MODEL_LABELS, label_first_ids)})
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        rec["error"] = "oom"
    with open(SCORES, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    n_done += 1
    if n_done in (20, 100) or n_done % 500 == 0:
        sec = (time.time() - t_run) / n_done
        peak = max([torch.cuda.max_memory_allocated(d) / 2**30 for d in range(torch.cuda.device_count())] or [0])
        print(f"{n_done}/{len(todo)} | {sec:.2f} s/item, about {(len(todo) - n_done) * sec / 3600:.1f} h left | "
              f"peak {peak:.1f} GiB | last: {condition}", flush=True)

print(f"finished {n_done} items this run", "(stopped at the time budget: commit again to continue)" if stopped_early else "", flush=True)

## Run

In [ ]:
cmd = [str(ENV_PY), "-u", "score_emotion_llama.py", "--repo", str(REPO), "--data-dir", str(DATA_DIR),
       "--out-dir", str(OUT_DIR), "--llama", str(LLAMA), "--ckpt", str(CKPT), "--hubert", str(HUBERT),
       "--model-name", MODEL_NAME, "--split-layer", str(SPLIT_LAYER), "--limit", str(LIMIT or 0),
       "--time-budget-h", str(TIME_BUDGET_H - (time.time() - T_START) / 3600),
       "--resume-glob", "/kaggle/input/**/emollama_runs/scores.jsonl"]
if os.environ.get("EMOLLAMA_CPU_TEST"):
    cmd.append("--cpu-test")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if "%|" not in line:  # leave out download progress bars
        print(line, end="", flush=True)
assert proc.wait() == 0, "the scoring script failed: see the lines above"

## Progress and quick checks

Six labels: `label_mass` is the probability the model puts on the first tokens of its six labels;
accuracy is against the majority label, on all clips and on clips whose majority label is not Disgust.

In [ ]:
SIX = ["Anger", "Fear", "Joy", "Neutral", "Sadness", "Surprise"]
manifest = pd.read_csv(DATA_DIR / "manifest.csv", dtype={"video_id": str, "clip_id": str})
s = pd.DataFrame([json.loads(l) for l in open(OUT_DIR / "scores.jsonl", encoding="utf-8") if l.strip()])
ok = s[s.get("error", pd.Series(index=s.index, dtype=object)).isna()] if len(s) else s
n_items = int((manifest["status"] == "ok").sum()) * 4 if not LIMIT else None
print(f"{len(ok)} items done" + (f" of {n_items}" if n_items else ""), "| OOM:", int((s.get("error") == "oom").sum()) if "error" in s else 0)
if len(ok):
    p = np.exp(ok[[f"logit_{l}" for l in SIX]].to_numpy(dtype=float) - ok[["lse"]].to_numpy())
    ok = ok.assign(label_mass=p.sum(1), pred=np.array(SIX)[p.argmax(1)]).merge(manifest[["uid", "ds_label"]], on="uid")
    ok["hit"] = ok["pred"] == ok["ds_label"]
    print(ok.groupby("condition").agg(n=("uid", "size"), label_mass=("label_mass", "mean"), accuracy=("hit", "mean"),
                                      accuracy_no_disgust=("hit", lambda x: x[ok.loc[x.index, "ds_label"] != "Disgust"].mean()),
                                      nonfinite=("finite", lambda x: int((~x.astype(bool)).sum())),
                                      gpu_s=("gpu_s", "mean"), prep_s=("prep_s", "mean"), tokens=("n_tokens", "mean")).round(3).to_string())
    print(ok.groupby("condition")["pred"].value_counts(normalize=True).unstack().round(2).to_string())